# Hito 1 — F1 Race Strategy Advisor: Baseline

**Team:** Ariel Van Kilsdonk & David Hernandez  
**Target:** `is_top10` (from official race-level file)  
**Split:** Train 2019–2021 | Calibration 2022 | Test 2023–2024  
**Primary metric:** Brier Score (lower = better)

**Note:** This notebook uses the official race-level file `f1_strategy_race_level.csv` as the data source. The current baseline is scenario-insensitive and serves as a risk baseline. Scenario-sensitive modeling will be implemented in Hito 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.calibration import calibration_curve

print('Libraries loaded.')

## 1. Load Data

We load the **race-level** CSV (`f1_strategy_race_level.csv`).  
The target `is_top10` is taken directly from the official file.

In [ ]:
# Adjust this path if running from a different directory
CSV_PATH = 'f1_strategy_race_level.csv'

race_level = pd.read_csv(CSV_PATH)
print(f'Loaded {len(race_level):,} race-level rows | seasons: {sorted(race_level["season"].unique())}')
race_level.head(3)

## 2. Feature Preparation

In [ ]:
def constructor_tier(team):
race_level['constructor_tier'] = race_level['Team'].apply(constructor_tier)
# Prepare constructor_tier if not present
def constructor_tier(team):
    if any(t in str(team) for t in ['Mercedes', 'Red Bull', 'Ferrari']):
        return 'top'
    elif any(t in str(team) for t in ['Williams', 'Haas', 'AlphaTauri', 'Alfa Romeo', 'Sauber']):
        return 'bottom'
    else:
        return 'mid'

if 'constructor_tier' not in race_level.columns:
    race_level['constructor_tier'] = race_level['Team'].apply(constructor_tier)

print(f'Race-level rows: {len(race_level):,}')
print(f'is_top10 distribution:\n{race_level["is_top10"].value_counts()}')
race_level.head(3)

## 3. Temporal Split (Locked)

| Block | Seasons | Purpose |
|---|---|---|
| Train | 2019, 2020, 2021 | Fit baseline rule |
| Calibration | 2022 | Fit calibration mapping (not used for model selection) |
| Test | 2023, 2024 | Final evaluation — look at once |

In [ ]:
train = race_level[race_level['season'].isin([2019, 2020, 2021])].copy()
calib = race_level[race_level['season'] == 2022].copy()
test  = race_level[race_level['season'].isin([2023, 2024])].copy()

print(f'Train rows: {len(train):,} | Calibration rows: {len(calib):,} | Test rows: {len(test):,}')
print(f'Train is_top10 rate: {train["is_top10"].mean():.3f}')
print(f'Test  is_top10 rate: {test["is_top10"].mean():.3f}')

## 4. Leakage Audit

This cell documents the role of every feature used in or excluded from the baseline.

In [ ]:
leakage_audit = pd.DataFrame([
    # Pre-race signals — safe to use as predictors
    {'feature': 'qualifying_position',   'type': 'pre-race signal',      'role': 'predictor',         'note': 'Proxy for grid_position. qualifying_time_s is empty — do NOT use as numeric signal.'},
    {'feature': 'constructor_tier',      'type': 'pre-race signal',      'role': 'predictor',         'note': 'Derived from team name; reflects prior-season car pace.'},
    {'feature': 'circuit',               'type': 'pre-race signal',      'role': 'predictor (future)','note': 'Circuit identity for Hito 2 circuit-specific models.'},

    # Scenario inputs — post-race observations used as user-controlled inputs
    {'feature': 'n_stops',               'type': 'scenario input',       'role': 'scenario input',    'note': 'Post-race observation allowed as scenario input. NOT a leaked predictor.'},
    {'feature': 'compounds_used',        'type': 'scenario input',       'role': 'scenario input',    'note': 'Post-race observation allowed as scenario input. NOT a leaked predictor.'},

    # Audit columns — post-race, used only for slicing/stress-testing
    {'feature': 'safety_car_race',       'type': 'audit column',         'role': 'audit/stress-test', 'note': 'Binary indicator. Not used as predictor (post-race). Used to slice results.'},
    {'feature': 'vsc_race',              'type': 'audit column',         'role': 'audit/stress-test', 'note': 'Same as above.'},
    {'feature': 'avg_lap_time',          'type': 'audit column',         'role': 'excluded',          'note': 'Post-race pace average. Would cause target leakage.'},

    # Target
    {'feature': 'is_top10',              'type': 'target',               'role': 'target',            'note': 'Derived from finishing_position ≤ 10.'},
    {'feature': 'finishing_position',    'type': 'target (raw)',          'role': 'excluded',          'note': 'Used only to build target. Not used as predictor.'},
])

print('=== LEAKAGE AUDIT ===')
leakage_audit

## 5. Heuristic Baseline (F1-Defendable)

**Rule:**
- P(top10) = 0.85 if qualifying_position ≤ 5
- P(top10) = 0.55 if 6 ≤ qualifying_position ≤ 10
- P(top10) = 0.20 otherwise

**Constructor tier modifier:** top → +0.05, bottom → −0.05, clipped to [0.05, 0.95]

**Rationale:** Starting position is the strongest single predictor of finishing position in F1. Constructor tier captures car pace independently of qualifying.

In [ ]:
def heuristic_proba(row):
    qp = row['qualifying_position']
    if qp <= 5:
        p = 0.85
    elif qp <= 10:
        p = 0.55
    else:
        p = 0.20

    tier = row['constructor_tier']
    if tier == 'top':
        p += 0.05
    elif tier == 'bottom':
        p -= 0.05

    return float(np.clip(p, 0.05, 0.95))

# Apply to train (to verify direction) and test (final evaluation)
train['pred_proba'] = train.apply(heuristic_proba, axis=1)
test['pred_proba']  = test.apply(heuristic_proba, axis=1)

print('Heuristic probabilities computed.')
print('Train sample:')
train[['Driver', 'circuit', 'season', 'qualifying_position', 'constructor_tier', 'pred_proba', 'is_top10']].head(8)

## 6. Evaluate Baseline on Test Set (2023–2024)

We look at the test set **once** here. Results are locked — we do not go back and tune the heuristic after seeing them.

In [ ]:
y_test      = test['is_top10'].values
y_pred_prob = test['pred_proba'].values

brier  = brier_score_loss(y_test, y_pred_prob)
logloss = log_loss(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print('=== BASELINE METRICS (Test 2023–2024) ===')
print(f'Brier Score : {brier:.4f}   (docent baseline: 0.132, grid-rule: 0.208)')
print(f'Log Loss    : {logloss:.4f}')
print(f'ROC-AUC     : {roc_auc:.4f}  (docent baseline: 0.892)')

print()
print('--- Comparison vs reference ---')
grid_rule_brier = 0.208
docent_brier    = 0.132
print(f'Beats grid-rule baseline (Brier < {grid_rule_brier}): {brier < grid_rule_brier}')
print(f'Beats or matches docent  (Brier < {docent_brier}):   {brier < docent_brier}')

if brier >= docent_brier:
    print()
    print('NOTE: Heuristic baseline does not match the calibrated docent model.')
    print('This is expected for a heuristic — Hito 2 experiments target Brier < 0.132.')

## 7. Calibration Curve

In [ ]:
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_pred_prob, n_bins=5, strategy='quantile'
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax.plot(mean_predicted_value, fraction_of_positives, 'o-', label='Heuristic baseline')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve — Heuristic Baseline (Test 2023–2024)')
ax.legend()
plt.tight_layout()
plt.savefig('calibration_curve_baseline.png', dpi=120)
plt.show()
print('Calibration curve saved to calibration_curve_baseline.png')

## 8. What-If Scenario Comparison

Demonstrating the scenario comparison use-case.  
**Note:** The current baseline is scenario-insensitive and does **not** use `n_stops` or `compound_sequence`. These features will be incorporated in Hito 2 ML models.

In [ ]:
scenarios = pd.DataFrame([
    # Scenario A — Monaco 2024, LEC
    {'label': 'LEC | Monaco 2024 | 1-stop M→H',   'qualifying_position': 1,  'constructor_tier': 'top'},
    {'label': 'LEC | Monaco 2024 | 2-stop S→M→H', 'qualifying_position': 1,  'constructor_tier': 'top'},
    # Scenario B — British GP 2023, STR
    {'label': 'STR | British 2023 | 1-stop M→H',  'qualifying_position': 8,  'constructor_tier': 'mid'},
    {'label': 'STR | British 2023 | 2-stop S→M→H','qualifying_position': 8,  'constructor_tier': 'mid'},
])

scenarios['P_top10'] = scenarios.apply(heuristic_proba, axis=1)

print('=== SCENARIO COMPARISON (Heuristic Baseline) ===')
print('Note: The current baseline does NOT react to scenario inputs (n_stops, compound_sequence). All scenario rows with the same qualifying_position and constructor_tier will get the same probability. Scenario sensitivity will be implemented in Hito 2.')
print()
print(scenarios[['label', 'qualifying_position', 'constructor_tier', 'P_top10']].to_string(index=False))

## 9. Reflection vs Docent Baseline

The heuristic rule serves as our **Hito 1 starting point**. Key observations:

1. **Direction is correct:** Higher grid → higher P(top10). F1-defendable without any data fitting.
2. **Calibration is coarse:** The rule assigns only 3 probability values, so the calibration curve shows stair-step behaviour rather than a smooth diagonal. This is expected for a heuristic.
3. **Comparison vs docent floor:** If Brier > 0.132, our Hito 2 experiments (Logistic Regression, LightGBM, Platt calibration) are designed to close this gap. We do not tune the heuristic after seeing test results.
4. **Scenario limitation:** The heuristic cannot differentiate between 1-stop and 2-stop strategies because it only uses grid position and constructor tier. This motivates the ML models in Hito 2 that incorporate scenario features.
5. **Honest recommendation caveat:** We do not recommend deploying the heuristic as a live tool unless it consistently achieves Brier ≤ 0.15 on held-out seasons not yet in this dataset.